In [67]:
import pandas as pd
import numpy as np

# Cargar precios ajustados
adj_close = pd.read_csv(
    "../Datos_csv/adj_close.csv",
    index_col=0,
    parse_dates=True
)

# Cargar volumen
volume = pd.read_csv(
    "../Datos_csv/volume.csv",
    index_col=0,
    parse_dates=True
)


In [68]:
to_drop = ['BIL', 'EWY', 'GLD', 'IAU', 'SHY']

adj_close = adj_close.drop(columns=to_drop)
volume    = volume.drop(columns=to_drop)


In [69]:
etf_class_map = {
    # 3.1.1 Mercado estadounidense (Core US)
    "SPY": "Core_US", "IVV": "Core_US", "VOO": "Core_US", "QQQ": "Core_US",
    "VTI": "Core_US", "ITOT": "Core_US", "DIA": "Core_US", "IWM": "Core_US",
    "IJR": "Core_US", "MDY": "Core_US",

    # 3.1.2 Factores y estilos
    "VUG": "Factor_Style", "IWF": "Factor_Style", "VTV": "Factor_Style",
    "IWD": "Factor_Style", "SCHD": "Factor_Style", "VIG": "Factor_Style",
    "DVY": "Factor_Style", "MTUM": "Factor_Style", "QUAL": "Factor_Style",
    "USMV": "Factor_Style", "VLUE": "Factor_Style",

    # 3.1.3 Sectores económicos
    "XLK": "Sector_economico", "XLF": "Sector_economico", "XLV": "Sector_economico", "XLY": "Sector_economico",
    "XLP": "Sector_economico", "XLE": "Sector_economico", "XLI": "Sector_economico", "XLU": "Sector_economico",
    "XLB": "Sector_economico", "XLRE": "Sector_economico",

    # 3.1.4 Mercados internacionales sin eeuu
    "VEA": "International", "IEFA": "International", "VWO": "International",
    "IEMG": "International", "EEM": "International", "EFA": "International",
    "EWJ": "International", "EWG": "International", "EWU": "International",
    "EWQ": "International", "INDA": "International", "EWZ": "International",
    "FXI": "International", "MCHI": "International", "EWT": "International",

    # 3.1.5 Renta fija (Bonos)
    "BND": "Bonds", "AGG": "Bonds", "IEF": "Bonds", "TLT": "Bonds",
    "LQD": "Bonds", "HYG": "Bonds", "JNK": "Bonds", "TIP": "Bonds",

    # 3.1.6 Materias primas y activos reales
    "SLV": "Real_Assets", "USO": "Real_Assets", "DBC": "Real_Assets",
    "VNQ": "Real_Assets"
}

In [70]:
# Retornos logarítmicos diarios
ret_log_diff = np.log(adj_close).diff()

# Cambio logarítmico del volumen diario
#si hay volúmenes 0, log(0) = -inf, luego limpiamos
vol_log_diff = np.log(volume.replace(0, np.nan)).diff()


In [71]:
ret_log_diff=ret_log_diff[1:]
vol_log_diff=vol_log_diff[1:]

In [72]:
# 4. PASAR A FORMATO LARGO CON pd.melt()
from pandas import wide_to_long


def formato_filas(df_columnas, nombre_variable):
    df_filas = (
        df_columnas
        .reset_index(names="Date")
        .melt(id_vars="Date", var_name="ETF", value_name=nombre_variable)
        .sort_values(["ETF", "Date"])
        .reset_index(drop=True)
    )
    return df_filas

ret_filas = formato_filas(ret_log_diff, "ret_log")
vol_filas = formato_filas(vol_log_diff, "vol_log_diff")

# Unir todo en un único df diario largo
df_diario = ret_filas.merge(vol_filas, on=["Date", "ETF"], how="left")

### Añadimos las clases

In [78]:
df_diario["clase"] = df_diario["ETF"].map(etf_class_map).fillna("Other")

In [ ]:
df_diario["clase"].value_counts()

In [44]:
# 3. variables temporales diarias
ret_log_diff["year"] = ret_log_diff.index.isocalendar().year.astype(int)
ret_log_diff = ret_log_diff.reset_index()

In [26]:
def crear_días_anteriores(df, lags=1):

    cols_no_etf = ["Date", "year"]
    etf_cols = [col for col in df.columns if col not in cols_no_etf]

    lag_dfs = []

    for lag in range(1, lags + 1):

        lag_df = df[etf_cols].shift(lag)
        lag_df = lag_df.add_suffix(f"_día_{lag}")

        lag_dfs.append(lag_df)

    df_lag = pd.concat([df] + lag_dfs, axis=1)

    return df_lag

crear semana -1

In [27]:
ret_log_diff_1 = crear_días_anteriores(ret_log_diff, lags=1)

print(ret_log_diff_1.head())

        Date       AGG       BND       DBC       DIA       DVY       EEM  \
0 2021-01-04       NaN       NaN       NaN       NaN       NaN       NaN   
1 2021-01-05 -0.001017 -0.001704  0.027658  0.005015  0.011437  0.023754   
2 2021-01-06 -0.004931 -0.004446  0.001330  0.014281  0.038779 -0.008487   
3 2021-01-07 -0.001023 -0.002058  0.004640  0.007402  0.003106  0.009425   
4 2021-01-08 -0.001195 -0.000917  0.007905  0.001738 -0.005116  0.026110   

        EFA       EWG       EWJ  ...  XLB_día_1  XLE_día_1  XLF_día_1  \
0       NaN       NaN       NaN  ...        NaN        NaN        NaN   
1  0.010309  0.004963  0.009244  ...        NaN        NaN        NaN   
2  0.011406  0.008933  0.011803  ...   0.022206   0.043810   0.004461   
3  0.001067  0.004895 -0.002055  ...   0.040107   0.030053   0.043216   
4  0.009286  0.004567  0.018205  ...   0.007442   0.014574   0.014323   

   XLI_día_1  XLK_día_1  XLP_día_1  XLRE_día_1  XLU_día_1  XLV_día_1  \
0        NaN        NaN        N

In [28]:
ret_log_diff_1 = ret_log_diff_1.iloc[1:]
ret_log_diff_1 = ret_log_diff_1.replace({None: np.nan})
print(ret_log_diff_1.head())

        Date       AGG       BND       DBC       DIA       DVY       EEM  \
1 2021-01-05 -0.001017 -0.001704  0.027658  0.005015  0.011437  0.023754   
2 2021-01-06 -0.004931 -0.004446  0.001330  0.014281  0.038779 -0.008487   
3 2021-01-07 -0.001023 -0.002058  0.004640  0.007402  0.003106  0.009425   
4 2021-01-08 -0.001195 -0.000917  0.007905  0.001738 -0.005116  0.026110   
5 2021-01-11 -0.001624 -0.001032 -0.007244 -0.002801  0.003113 -0.013618   

        EFA       EWG       EWJ  ...  XLB_día_1  XLE_día_1  XLF_día_1  \
1  0.010309  0.004963  0.009244  ...        NaN        NaN        NaN   
2  0.011406  0.008933  0.011803  ...   0.022206   0.043810   0.004461   
3  0.001067  0.004895 -0.002055  ...   0.040107   0.030053   0.043216   
4  0.009286  0.004567  0.018205  ...   0.007442   0.014574   0.014323   
5 -0.013159 -0.016848 -0.009425  ...  -0.004824  -0.001207  -0.000647   

   XLI_día_1  XLK_día_1  XLP_día_1  XLRE_día_1  XLU_día_1  XLV_día_1  \
1        NaN        NaN        N

crear semana -2

In [29]:
ret_log_diff_2 = crear_días_anteriores(ret_log_diff, lags=2)

print(ret_log_diff_2.head())

        Date       AGG       BND       DBC       DIA       DVY       EEM  \
0 2021-01-04       NaN       NaN       NaN       NaN       NaN       NaN   
1 2021-01-05 -0.001017 -0.001704  0.027658  0.005015  0.011437  0.023754   
2 2021-01-06 -0.004931 -0.004446  0.001330  0.014281  0.038779 -0.008487   
3 2021-01-07 -0.001023 -0.002058  0.004640  0.007402  0.003106  0.009425   
4 2021-01-08 -0.001195 -0.000917  0.007905  0.001738 -0.005116  0.026110   

        EFA       EWG       EWJ  ...  XLB_día_2  XLE_día_2  XLF_día_2  \
0       NaN       NaN       NaN  ...        NaN        NaN        NaN   
1  0.010309  0.004963  0.009244  ...        NaN        NaN        NaN   
2  0.011406  0.008933  0.011803  ...        NaN        NaN        NaN   
3  0.001067  0.004895 -0.002055  ...   0.022206   0.043810   0.004461   
4  0.009286  0.004567  0.018205  ...   0.040107   0.030053   0.043216   

   XLI_día_2  XLK_día_2  XLP_día_2  XLRE_día_2  XLU_día_2  XLV_día_2  \
0        NaN        NaN        N

In [30]:
ret_log_diff_2 = ret_log_diff_2.iloc[1:]
ret_log_diff_2 = ret_log_diff_2.replace({None: np.nan})
print(ret_log_diff_2.head())

        Date       AGG       BND       DBC       DIA       DVY       EEM  \
1 2021-01-05 -0.001017 -0.001704  0.027658  0.005015  0.011437  0.023754   
2 2021-01-06 -0.004931 -0.004446  0.001330  0.014281  0.038779 -0.008487   
3 2021-01-07 -0.001023 -0.002058  0.004640  0.007402  0.003106  0.009425   
4 2021-01-08 -0.001195 -0.000917  0.007905  0.001738 -0.005116  0.026110   
5 2021-01-11 -0.001624 -0.001032 -0.007244 -0.002801  0.003113 -0.013618   

        EFA       EWG       EWJ  ...  XLB_día_2  XLE_día_2  XLF_día_2  \
1  0.010309  0.004963  0.009244  ...        NaN        NaN        NaN   
2  0.011406  0.008933  0.011803  ...        NaN        NaN        NaN   
3  0.001067  0.004895 -0.002055  ...   0.022206   0.043810   0.004461   
4  0.009286  0.004567  0.018205  ...   0.040107   0.030053   0.043216   
5 -0.013159 -0.016848 -0.009425  ...   0.007442   0.014574   0.014323   

   XLI_día_2  XLK_día_2  XLP_día_2  XLRE_día_2  XLU_día_2  XLV_día_2  \
1        NaN        NaN        N

In [31]:
ret_log_diff_3 = crear_días_anteriores(ret_log_diff, lags=3)
ret_log_diff_3 = ret_log_diff_3.iloc[1:]
ret_log_diff_3 = ret_log_diff_3.replace({None: np.nan})

print(ret_log_diff_3.head())

        Date       AGG       BND       DBC       DIA       DVY       EEM  \
1 2021-01-05 -0.001017 -0.001704  0.027658  0.005015  0.011437  0.023754   
2 2021-01-06 -0.004931 -0.004446  0.001330  0.014281  0.038779 -0.008487   
3 2021-01-07 -0.001023 -0.002058  0.004640  0.007402  0.003106  0.009425   
4 2021-01-08 -0.001195 -0.000917  0.007905  0.001738 -0.005116  0.026110   
5 2021-01-11 -0.001624 -0.001032 -0.007244 -0.002801  0.003113 -0.013618   

        EFA       EWG       EWJ  ...  XLB_día_3  XLE_día_3  XLF_día_3  \
1  0.010309  0.004963  0.009244  ...        NaN        NaN        NaN   
2  0.011406  0.008933  0.011803  ...        NaN        NaN        NaN   
3  0.001067  0.004895 -0.002055  ...        NaN        NaN        NaN   
4  0.009286  0.004567  0.018205  ...   0.022206   0.043810   0.004461   
5 -0.013159 -0.016848 -0.009425  ...   0.040107   0.030053   0.043216   

   XLI_día_3  XLK_día_3  XLP_día_3  XLRE_día_3  XLU_día_3  XLV_día_3  \
1        NaN        NaN        N

In [32]:
ret_log_diff_4 = crear_días_anteriores(ret_log_diff, lags=4)
ret_log_diff_4 = ret_log_diff_4.iloc[1:]
ret_log_diff_4 = ret_log_diff_4.replace({None: np.nan})

print(ret_log_diff_4.head())

        Date       AGG       BND       DBC       DIA       DVY       EEM  \
1 2021-01-05 -0.001017 -0.001704  0.027658  0.005015  0.011437  0.023754   
2 2021-01-06 -0.004931 -0.004446  0.001330  0.014281  0.038779 -0.008487   
3 2021-01-07 -0.001023 -0.002058  0.004640  0.007402  0.003106  0.009425   
4 2021-01-08 -0.001195 -0.000917  0.007905  0.001738 -0.005116  0.026110   
5 2021-01-11 -0.001624 -0.001032 -0.007244 -0.002801  0.003113 -0.013618   

        EFA       EWG       EWJ  ...  XLB_día_4  XLE_día_4  XLF_día_4  \
1  0.010309  0.004963  0.009244  ...        NaN        NaN        NaN   
2  0.011406  0.008933  0.011803  ...        NaN        NaN        NaN   
3  0.001067  0.004895 -0.002055  ...        NaN        NaN        NaN   
4  0.009286  0.004567  0.018205  ...        NaN        NaN        NaN   
5 -0.013159 -0.016848 -0.009425  ...   0.022206    0.04381   0.004461   

   XLI_día_4  XLK_día_4  XLP_día_4  XLRE_día_4  XLU_día_4  XLV_día_4  \
1        NaN        NaN        N

In [33]:
ret_log_diff_5 = crear_días_anteriores(ret_log_diff, lags=5)
ret_log_diff_5 = ret_log_diff_5.iloc[1:]
ret_log_diff_5 = ret_log_diff_5.replace({None: np.nan})

print(ret_log_diff_5.head())

        Date       AGG       BND       DBC       DIA       DVY       EEM  \
1 2021-01-05 -0.001017 -0.001704  0.027658  0.005015  0.011437  0.023754   
2 2021-01-06 -0.004931 -0.004446  0.001330  0.014281  0.038779 -0.008487   
3 2021-01-07 -0.001023 -0.002058  0.004640  0.007402  0.003106  0.009425   
4 2021-01-08 -0.001195 -0.000917  0.007905  0.001738 -0.005116  0.026110   
5 2021-01-11 -0.001624 -0.001032 -0.007244 -0.002801  0.003113 -0.013618   

        EFA       EWG       EWJ  ...  XLB_día_5  XLE_día_5  XLF_día_5  \
1  0.010309  0.004963  0.009244  ...        NaN        NaN        NaN   
2  0.011406  0.008933  0.011803  ...        NaN        NaN        NaN   
3  0.001067  0.004895 -0.002055  ...        NaN        NaN        NaN   
4  0.009286  0.004567  0.018205  ...        NaN        NaN        NaN   
5 -0.013159 -0.016848 -0.009425  ...        NaN        NaN        NaN   

   XLI_día_5  XLK_día_5  XLP_día_5  XLRE_día_5  XLU_día_5  XLV_día_5  \
1        NaN        NaN        N

In [34]:
ret_log_diff_6 = crear_días_anteriores(ret_log_diff, lags=6)
ret_log_diff_6 = ret_log_diff_6.iloc[1:]
ret_log_diff_6 = ret_log_diff_6.replace({None: np.nan})

print(ret_log_diff_6.head())

        Date       AGG       BND       DBC       DIA       DVY       EEM  \
1 2021-01-05 -0.001017 -0.001704  0.027658  0.005015  0.011437  0.023754   
2 2021-01-06 -0.004931 -0.004446  0.001330  0.014281  0.038779 -0.008487   
3 2021-01-07 -0.001023 -0.002058  0.004640  0.007402  0.003106  0.009425   
4 2021-01-08 -0.001195 -0.000917  0.007905  0.001738 -0.005116  0.026110   
5 2021-01-11 -0.001624 -0.001032 -0.007244 -0.002801  0.003113 -0.013618   

        EFA       EWG       EWJ  ...  XLB_día_6  XLE_día_6  XLF_día_6  \
1  0.010309  0.004963  0.009244  ...        NaN        NaN        NaN   
2  0.011406  0.008933  0.011803  ...        NaN        NaN        NaN   
3  0.001067  0.004895 -0.002055  ...        NaN        NaN        NaN   
4  0.009286  0.004567  0.018205  ...        NaN        NaN        NaN   
5 -0.013159 -0.016848 -0.009425  ...        NaN        NaN        NaN   

   XLI_día_6  XLK_día_6  XLP_día_6  XLRE_día_6  XLU_día_6  XLV_día_6  \
1        NaN        NaN        N

In [35]:
ret_log_diff_7 = crear_días_anteriores(ret_log_diff, lags=7)
ret_log_diff_7 = ret_log_diff_7.iloc[1:]
ret_log_diff_7 = ret_log_diff_7.replace({None: np.nan})

print(ret_log_diff_7.head())

        Date       AGG       BND       DBC       DIA       DVY       EEM  \
1 2021-01-05 -0.001017 -0.001704  0.027658  0.005015  0.011437  0.023754   
2 2021-01-06 -0.004931 -0.004446  0.001330  0.014281  0.038779 -0.008487   
3 2021-01-07 -0.001023 -0.002058  0.004640  0.007402  0.003106  0.009425   
4 2021-01-08 -0.001195 -0.000917  0.007905  0.001738 -0.005116  0.026110   
5 2021-01-11 -0.001624 -0.001032 -0.007244 -0.002801  0.003113 -0.013618   

        EFA       EWG       EWJ  ...  XLB_día_7  XLE_día_7  XLF_día_7  \
1  0.010309  0.004963  0.009244  ...        NaN        NaN        NaN   
2  0.011406  0.008933  0.011803  ...        NaN        NaN        NaN   
3  0.001067  0.004895 -0.002055  ...        NaN        NaN        NaN   
4  0.009286  0.004567  0.018205  ...        NaN        NaN        NaN   
5 -0.013159 -0.016848 -0.009425  ...        NaN        NaN        NaN   

   XLI_día_7  XLK_día_7  XLP_día_7  XLRE_día_7  XLU_día_7  XLV_día_7  \
1        NaN        NaN        N

In [36]:
ret_log_diff_8 = crear_días_anteriores(ret_log_diff, lags=8)
ret_log_diff_8 = ret_log_diff_8.iloc[1:]
ret_log_diff_8 = ret_log_diff_8.replace({None: np.nan})

print(ret_log_diff_8.head())

        Date       AGG       BND       DBC       DIA       DVY       EEM  \
1 2021-01-05 -0.001017 -0.001704  0.027658  0.005015  0.011437  0.023754   
2 2021-01-06 -0.004931 -0.004446  0.001330  0.014281  0.038779 -0.008487   
3 2021-01-07 -0.001023 -0.002058  0.004640  0.007402  0.003106  0.009425   
4 2021-01-08 -0.001195 -0.000917  0.007905  0.001738 -0.005116  0.026110   
5 2021-01-11 -0.001624 -0.001032 -0.007244 -0.002801  0.003113 -0.013618   

        EFA       EWG       EWJ  ...  XLB_día_8  XLE_día_8  XLF_día_8  \
1  0.010309  0.004963  0.009244  ...        NaN        NaN        NaN   
2  0.011406  0.008933  0.011803  ...        NaN        NaN        NaN   
3  0.001067  0.004895 -0.002055  ...        NaN        NaN        NaN   
4  0.009286  0.004567  0.018205  ...        NaN        NaN        NaN   
5 -0.013159 -0.016848 -0.009425  ...        NaN        NaN        NaN   

   XLI_día_8  XLK_día_8  XLP_día_8  XLRE_día_8  XLU_día_8  XLV_día_8  \
1        NaN        NaN        N

In [37]:
import os

def guardar_dataset(df, nombre_archivo):
    
    out_dir = "../Datos_csv"
    os.makedirs(out_dir, exist_ok=True)

    path = os.path.join(out_dir, nombre_archivo)

    df.to_csv(path, index=False)

    print("Guardado:")
    print(" -", path, df.shape)

In [38]:
guardar_dataset(ret_log_diff, "df_diario.csv")
guardar_dataset(ret_log_diff_1, "df_diario_1.csv")
guardar_dataset(ret_log_diff_2, "df_diario_2.csv")
guardar_dataset(ret_log_diff_3, "df_diario_3.csv")
guardar_dataset(ret_log_diff_4, "df_diario_4.csv")
guardar_dataset(ret_log_diff_5, "df_diario_5.csv")
guardar_dataset(ret_log_diff_6, "df_diario_6.csv")
guardar_dataset(ret_log_diff_7, "df_diario_7.csv")
guardar_dataset(ret_log_diff_8, "df_diario_8.csv")

Guardado:
 - ../Datos_csv\df_diario.csv (1276, 60)
Guardado:
 - ../Datos_csv\df_diario_1.csv (1275, 118)
Guardado:
 - ../Datos_csv\df_diario_2.csv (1275, 176)
Guardado:
 - ../Datos_csv\df_diario_3.csv (1275, 234)
Guardado:
 - ../Datos_csv\df_diario_4.csv (1275, 292)
Guardado:
 - ../Datos_csv\df_diario_5.csv (1275, 350)
Guardado:
 - ../Datos_csv\df_diario_6.csv (1275, 408)
Guardado:
 - ../Datos_csv\df_diario_7.csv (1275, 466)
Guardado:
 - ../Datos_csv\df_diario_8.csv (1275, 524)
